<a href="https://colab.research.google.com/github/Chelladuraimct/SQL-Intermediate-Querying-Subjective/blob/main/SQL_Intermediate_Querying_Subjective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
**Intermediate SQL Querying Assignment**

**Exploring Customer Orders with SQL Aggregation and Joins**

---

**Import Libraries and Load Dataset**

In [ ]:
import pandas as pd
import sqlite3

customers_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/customers.csv"
orders_url = "https://raw.githubusercontent.com/graphql-compose/graphql-compose-examples/master/examples/northwind/data/csv/orders.csv"

# Load datasets
customers_df = pd.read_csv(customers_url)
orders_df = pd.read_csv(orders_url)

# Create SQLite in-memory database
conn = sqlite3.connect(":memory:")

# Load data into SQL tables
customers_df.to_sql("customers", conn, index=False, if_exists="replace")
orders_df.to_sql("orders", conn, index=False, if_exists="replace")

print("Tables loaded successfully")

Tables loaded successfully


**Task 1 — Aggregation and Grouping**

**SQL Query**

In [ ]:
query1 = """
SELECT
    CustomerID,
    COUNT(OrderID) AS order_count,
    SUM(Freight) AS total_freight,
    AVG(Freight) AS avg_freight
FROM orders
GROUP BY CustomerID
ORDER BY total_freight DESC
"""

result1 = pd.read_sql_query(query1, conn)

result1.head(10)

,customerID,order_count,total_freight,avg_freight
0,SAVEA,31,6683.70,215.603226
1,ERNSH,30,6205.39,206.846333
2,QUICK,28,5605.63,200.201071
3,HUNGO,19,2755.24,145.012632
4,RATTC,18,2134.21,118.567222
5,QUEEN,13,1982.70,152.515385
6,FOLKO,19,1678.08,88.320000
7,BERGS,18,1559.52,86.640000
8,FRANK,15,1403.44,93.562667
9,MEREP,13,1394.22,107.247692


**Task 2 — WHERE vs HAVING**

**Query A — Using WHERE (Filter Before Aggregation)**

In [ ]:
queryA = """
SELECT
    CustomerID,
    COUNT(OrderID) AS high_freight_orders
FROM orders
WHERE Freight > 50
GROUP BY CustomerID
"""

resultA = pd.read_sql_query(queryA, conn)

resultA.head()

,customerID,high_freight_orders
0,ALFKI,2
1,ANTON,2
2,AROUT,2
3,BERGS,11
4,BLAUS,1


**Query B — Using HAVING (Filter After Aggregation)**

In [ ]:
queryB = """
SELECT
    CustomerID,
    SUM(Freight) AS total_freight
FROM orders
GROUP BY CustomerID
HAVING SUM(Freight) > 500
"""

resultB = pd.read_sql_query(queryB, conn)

resultB.head()

,customerID,total_freight
0,BERGS,1559.52
1,BLONP,623.66
2,BONAP,1357.87
3,BOTTM,793.95
4,EASTC,832.34


**Explanation:**

WHERE filters rows before the grouping operation occurs.
In Query A, only orders where Freight > 50 are included before calculating the count of orders per customer.

HAVING, however, filters after the aggregation step.
In Query B, the database first calculates the total freight per customer, and then returns only those customers whose total freight exceeds 500.

**Task 3 — JOIN and Aggregation**

**Query 1 — INNER JOIN (Customers With Orders Only)**

In [ ]:
query3_inner = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    SUM(o.Freight) AS total_freight
FROM customers c
INNER JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CompanyName, c.Country
ORDER BY total_freight DESC
"""

result3_inner = pd.read_sql_query(query3_inner, conn)

result3_inner.head()

,companyName,country,order_count,total_freight
0,Save-a-lot Markets,USA,31,6683.70
1,Ernst Handel,Austria,30,6205.39
2,QUICK-Stop,Germany,28,5605.63
3,Hungry Owl All-Night Grocers,Ireland,19,2755.24
4,Rattlesnake Canyon Grocery,USA,18,2134.21


**Query 2 — LEFT JOIN (Include All Customers)**

In [ ]:
query3_left = """
SELECT
    c.CompanyName,
    c.Country,
    COUNT(o.OrderID) AS order_count,
    COALESCE(SUM(o.Freight),0) AS total_freight
FROM customers c
LEFT JOIN orders o
ON c.CustomerID = o.CustomerID
GROUP BY c.CompanyName, c.Country
ORDER BY total_freight DESC
"""

result3_left = pd.read_sql_query(query3_left, conn)

result3_left.head()

,companyName,country,order_count,total_freight
0,Save-a-lot Markets,USA,31,6683.70
1,Ernst Handel,Austria,30,6205.39
2,QUICK-Stop,Germany,28,5605.63
3,Hungry Owl All-Night Grocers,Ireland,19,2755.24
4,Rattlesnake Canyon Grocery,USA,18,2134.21


**Explanation:**

The INNER JOIN query returns only customers who have matching records in the orders table, meaning customers who have placed at least one order.

The LEFT JOIN query includes all customers, even those without any orders. For customers with no orders, the order fields become NULL, and we use COALESCE() to convert the total freight to 0 for easier interpretation.